# Run 1 (1 year): inject the best-fit wavelet modulation and recover it with RJMCMC

Same test as the 30-day [run 1](../../30_days/run1/run1_inject_recover.ipynb), over **one year** of data. Check that
the reversible-jump sampler recovers a modulation that the wavelet model can represent **exactly**. The injection is the
least-squares wavelet fit of the true GF envelope (sin ψ = −0.83) over the year
([fit_reference.py](../../common/fit_reference.py), `outputs/reference/atoms.npz`), written in the inference
parametrisation ($b_A\equiv 0$ absorbed into the galactic amplitude):

$$\log P_A(t) = \sum_{k=1}^{K_A} a_k\,\psi\left(\tfrac{t-t_k}{\tau_k}\right),\qquad
\log P_E(t) = \ell_E + \sum_{k=1}^{K_E} a_k\,\psi\left(\tfrac{t-t_k}{\tau_k}\right),\qquad \log_{10}A_{\rm gal} = \texttt{amp\_eff}.$$

Everything else is as in the 30-day run: spectra and band of the paper-2 pack-2 run, TDI gen 1, $\Delta t$ = 10 s,
seed 2026, 2 extra days simulated at each end and cut, $N_f = 3600$ (10-h pixels). The code is the shared
[simulations/common/](../../common/) code; only the config differs, [config/wavelet.yaml](../config/wavelet.yaml)
(the 30-day one is [here](../../30_days/config/wavelet.yaml)):

* `T` = 365.25 d, trimmed by the WDM grid to an even number of 10-h pixels: 876 pixels = 365.0 d.
* Atom widths 5–120 d (30 days: 5–60 d) and up to 16 atoms per channel (30 days: 8). The envelope repeats twice a
  year, so a year needs many more atoms than 30 days; the reference fit below tries up to `K_MAX_FIT` atoms and keeps the
  smallest K with an rms error below 1%.
* Outputs go to `simulations/1_year/outputs/` (`reference/`, `run1/`).

The likelihood grid is about 12 times larger than for 30 days, so expect each sampler step to take about 12 times
longer (the 30-day run took about 40 min on 8 cores). Set `RERUN = True` to fit, simulate and sample; otherwise the
saved files in `outputs/` are loaded.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
HERE = ROOT / 'simulations' / '1_year'
sys.path.insert(0, str(ROOT / 'simulations' / 'common'))
import injection as I
import fit_reference as FR   # sets the Agg backend: switched back to inline below
from bahamas.wavelet.model import WaveletData, DAY
from bahamas.wavelet import plotting
from bahamas.wavelet.simulate import power_modulation
import diagnostics as D
%matplotlib inline

RERUN = False
RUN_K_TRACE = False      # rerun with the burn-in stored (section 3), as long again as the run itself
K_MAX_FIT = 12          # most atoms per channel the reference fit tries

CFG = I.load_config(HERE / 'config' / 'wavelet.yaml')   # 1-year window, wider and more atoms, own outputs/

OUT = HERE / 'outputs' / 'run1'
OUT.mkdir(parents=True, exist_ok=True)
CH = ['A', 'E']

C_TRUTH, C_POST, C_SECOND, C_MUTED, C_BAND = '#0b0b0b', '#2a78d6', '#eb6834', '#52514e', '#86b6ef'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False,
                     'axes.spines.right': False, 'lines.linewidth': 1.5, 'legend.frameon': False})

cfg = CFG
print({k: v for k, v in cfg.items() if k not in ('wavelet', 'inference')})
print('wavelet  :', cfg['wavelet'])
print('inference:', cfg['inference'])

## 0. The reference fit (the injection)

Wavelet fit of the true envelope $M_c^2(t)$ over the year, for K = 0 … `K_MAX_FIT` atoms per channel; the smallest K
with an rms error below 1% is kept and becomes the injection. The printout checks every injected value against the
inference prior: if K hits `K_MAX_FIT` without reaching 1%, raise `K_MAX_FIT` (and `max_wavelets` if needed).

In [ ]:
ref_path = Path(I.reference_path(CFG))
if RERUN or not ref_path.exists():
    FR.run(config=CFG, k_max=K_MAX_FIT)
r_ref = np.load(ref_path)
print(f"K: A = {len(r_ref['atoms_A'])}, E = {len(r_ref['atoms_E'])};  rms: A {float(r_ref['rms_A']):.4f}, E {float(r_ref['rms_E']):.4f}")
print('rms by K (rows A, E):\n', np.round(r_ref['rms_scan'], 4))
FR.check_prior(dict(r_ref), CFG)
display(Image(str(ref_path.parent / 'fit.png'), width=900))

## 1. The injection

The atoms were fitted on the span the posterior uses, $[0, N\,\Delta t] = [0, 365\,{\rm d}]$, so no remapping is needed.
Shown for context: the true GF envelope $M_c^2(t)$, rescaled by $10^{\log_{10}A_{\rm true}-\log_{10}A_{\rm eff}}$ so that it is
on the same scale as the injection (their rms difference is printed in section 0). The shaded edges are the pixels the
likelihood drops (`edge_bins = 2`, 20 h at each end: the transform is periodic; see [data_checks.py](../../30_days/checks/data_checks.py)).

Lower panels: the atoms themselves in the time domain, i.e. each term $a_k\,\psi\big((t-t_k)/\tau_k\big)$ of
$\ln P_c(t)$ with the Gaussian atom $\psi(x) = e^{-x^2/2}$, drawn in $\ln P$ units (the units in which they add).
Their sum plus the level $b_c$ ($b_A \equiv 0$, $b_E$ = `level_E`) is $\ln P_c(t)$ (black); exponentiating it gives the
curve in the upper panel. The lower panels extend to $\pm 2.5\,\tau_k$ around each centre so the full atom
shape shows; the 1-year data span is the unshaded part. The dotted lines are the atom centres $t_k$.

In [ ]:
ref = I.load_reference(I.reference_path(CFG))
data, inj = I.load_or_simulate('wavelet', 'run1', CFG, rerun=RERUN)
like, post = I.build_posterior(data, CFG)
print(f'WDM grid: Nt = {data.Nt} x Nf = {data.Nf} (dT = {data.Nf * data.dt / 3600:.1f} h); likelihood: '
      f'{len(like.nu)} frequency groups x {len(like.times)} time groups x {len(data.channels)} channels')
print(f'analysed times: {like.times[0] / DAY:.2f} - {like.times[-1] / DAY:.2f} d (group centres)')

amp_true = I.true_amp()
print(f"amp: true GF {amp_true:.3f}, effective (injected) {ref['amp']:.3f};  level_E = {ref['level_E']:.3f}")
print('\ninjected atoms (posterior convention):')
for c in CH:
    for a in inj.atoms[c]:
        print(f'  {c}: centre {a[0] * inj.wm.T / DAY:7.2f} d (t_frac {a[0]:.4f}), width {10 ** a[1]:5.1f} d, amp {a[2]:+.3f}')

In [ ]:
t_fine = np.linspace(0, inj.wm.T, 2000)
scale = 10 ** (amp_true - ref['amp'])
dT = data.Nf * data.dt
eb = cfg['wavelet']['edge_bins']
T_d = inj.wm.T / DAY
C_ATOM = list(plt.cm.tab10.colors)

fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), gridspec_kw={'height_ratios': [1.2, 1]})
for i, c in enumerate(CH):
    ax, axa = axes[0, i], axes[1, i]
    M2 = np.asarray(power_modulation(I.sky_params(), t_fine, i))
    ax.plot(t_fine / DAY, scale * M2, color=C_MUTED, lw=3, alpha=0.5, label=r'true GF $M_c^2$ (rescaled)')
    ax.plot(t_fine / DAY, np.asarray(inj.P(t_fine, i)), color=C_TRUTH, label='injected wavelet $P_c(t)$')
    ax.set_title(f'channel {c}  (dotted: atom centres)')
    ax.set_xlabel('time [days]')
    for a in inj.atoms[c]:
        ax.axvline(a[0] * T_d, color=C_SECOND, ls=':', lw=1)
    ax.axvspan(0, eb * dT / DAY, color=C_MUTED, alpha=0.12, lw=0)
    ax.axvspan(T_d - eb * dT / DAY, T_d, color=C_MUTED, alpha=0.12, lw=0)

    # the atoms themselves, a_k psi((t - t_k) / tau_k) in ln P units, over +-2.5 widths so their full shape shows;
    # the data span [0, T] is the unshaded part
    cen, wid = inj.atoms[c][:, 0] * T_d, 10 ** inj.atoms[c][:, 1]
    lo, hi = min(0, (cen - 2.5 * wid).min()), max(T_d, (cen + 2.5 * wid).max())
    t_wide = np.linspace(lo, hi, 3000) * DAY
    for k, a in enumerate(inj.atoms[c]):
        axa.plot(t_wide / DAY, np.asarray(inj.wm.log_modulation(t_wide, a[None])), color=C_ATOM[k % len(C_ATOM)], lw=1.6,
                 label=rf'atom {k + 1}: $t_k$ = {cen[k]:.1f} d, $\tau_k$ = {wid[k]:.1f} d, $a_k$ = {a[2]:+.2f}')
        axa.axvline(cen[k], color=C_SECOND, ls=':', lw=1)
    level = inj.levels[c]
    if level:
        axa.axhline(level, color=C_MUTED, lw=1, ls='--', label=rf'level $b_{c}$ = {level:+.2f}')
    axa.plot(t_wide / DAY, np.asarray(inj.wm.log_modulation(t_wide, inj.atoms[c], level=level)), color=C_TRUTH,
             lw=1.2, label=r'sum: $\ln P_c(t)$')
    axa.axhline(0, color=C_MUTED, lw=0.6)
    axa.axvspan(lo, 0, color=C_MUTED, alpha=0.12, lw=0)
    axa.axvspan(T_d, hi, color=C_MUTED, alpha=0.12, lw=0)
    axa.text(T_d / 2, 0.03, 'data span', transform=axa.get_xaxis_transform(), ha='center', va='bottom', fontsize=8,
             color=C_MUTED)
    axa.set_xlim(lo, hi)
    axa.set_xlabel('time [days]')
    axa.legend(fontsize=7.5, loc='upper center', bbox_to_anchor=(0.5, -0.3), ncol=2)
axes[0, 0].set_ylabel(r'$P_c(t)$')
axes[1, 0].set_ylabel(r'$a_k\,\psi((t-t_k)/\tau_k)$  [ln P]')
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'injection.png', dpi=150)

The printout above lists the injected atoms. Atoms within a few widths of the window ends are only partly seen by the
data, as in the 30-day run.

In [ ]:
plotting.plot_spectrogram(data, cfg['f1'], cfg['f2'], str(OUT / 'data_spectrogram.png'))
display(Image(str(OUT / 'data_spectrogram.png'), width=700))

### The data in the time-frequency plane, as the likelihood sees it

The WDM transform turns each channel's 1-year series into a grid of $N_t \times N_f = 876 \times 3600$ real
coefficients $w_{c,nm}$: columns are 10-h time pixels, rows are $13.9\,\mu$Hz frequency pixels
([GF_TIME_FREQUENCY_NOTE](../../../reports/GF_TIME_FREQUENCY_NOTE.pdf), Part II). Per channel:

* **left: every pixel** in the analysed band, as $2\Delta t\,w^2_{c,nm}$ (units of a one-sided PSD, so its expectation is
  the local PSD). Single pixels scatter as $\chi^2_1$, hence the speckle. Hatched: the 2 + 2 edge pixels the likelihood
  drops.
* **middle: the likelihood grid**, $2\Delta t\,Y_{c,sg}/\nu_g$: pixels summed in blocks of $n_\tau = 2$ time pixels (20 h)
  and log-spaced frequency groups (sizes printed below). These numbers are all the likelihood uses.
* **right: the same grid divided by each frequency group's mean over time.** The spectrum is gone, and what is left is
  the time dependence: the galactic modulation where the galaxy dominates the noise (roughly 0.3–3 mHz), plus
  $\chi^2_{\nu_g}$ scatter, which is large at low frequency where groups hold few pixels.

Because every column is a separate set of numbers, a gap in the data maps onto a range of columns. Note that each WDM
basis function extends over a few $\Delta T$ in time, so the pixels next to a gap are affected too, as at the edges.

In [ ]:
from bahamas.wavelet.model import frequency_groups
band = like.pixels                                   # frequency pixels in [f1, f2]
f_pix = data.freqs[band]
dF = data.freqs[1] - data.freqs[0]
f_pix_edges = np.append(f_pix - dF / 2, f_pix[-1] + dF / 2)
t_pix_edges = np.arange(data.Nt + 1) * dT / DAY
blk = like.time_index.reshape(-1, like.time_bin)       # time pixels of each block
t_blk_edges = np.append(data.times[blk[:, 0]], data.times[blk[-1, -1]] + dT) / DAY
_, starts = frequency_groups(data.freqs, cfg['f1'], cfg['f2'], cfg['wavelet']['freq_bins'])
f_grp_edges = np.append(f_pix[starts] - dF / 2, f_pix[-1] + dF / 2)
nu = np.asarray(like.nu)
Ybar = 2 * data.dt * np.asarray(like.Y) / nu            # (channel, block, group), PSD units

fig, axes = plt.subplots(2, 3, figsize=(15, 7.4), sharey=True)
for i, c in enumerate(CH):
    pix = 2 * data.dt * data.w[i][:, band] ** 2
    lo, hi = np.percentile(np.log10(Ybar[i]), [0.5, 99.5])
    m0 = axes[i, 0].pcolormesh(t_pix_edges, f_pix_edges, np.log10(pix.T), vmin=lo - 1, vmax=hi, cmap='viridis',
                               shading='flat', rasterized=True)
    m1 = axes[i, 1].pcolormesh(t_blk_edges, f_grp_edges, np.log10(Ybar[i].T), vmin=lo, vmax=hi, cmap='viridis',
                               shading='flat', rasterized=True)
    rel = Ybar[i] / Ybar[i].mean(axis=0, keepdims=True)
    m2 = axes[i, 2].pcolormesh(t_blk_edges, f_grp_edges, np.log10(rel.T), vmin=-0.3, vmax=0.3, cmap='RdBu_r',
                               shading='flat', rasterized=True)
    for ax in axes[i, 0], axes[i, 1]:
        for x0, x1 in [(0, eb * dT / DAY), (T_d - eb * dT / DAY, T_d)]:
            ax.axvspan(x0, x1, facecolor='white', alpha=0.6, hatch='///', edgecolor=C_MUTED, lw=0)
    for ax in axes[i]:
        ax.set_yscale('log')
        ax.set_xlim(0, T_d)
        ax.grid(False)
    axes[i, 0].set_ylabel(f'channel {c}\nfrequency [Hz]')
    fig.colorbar(m0, ax=axes[i, 0], label=r'$\log_{10}\,2\Delta t\,w^2_{nm}$ [1/Hz]')
    fig.colorbar(m1, ax=axes[i, 1], label=r'$\log_{10}\,2\Delta t\,Y_{sg}/\nu_g$ [1/Hz]')
    fig.colorbar(m2, ax=axes[i, 2], label=r'$\log_{10}$ (grid / its time mean)')
for ax, title in zip(axes[0], [f'every WDM pixel ({data.Nt} x {len(band)} in band)',
                               f'likelihood grid ({len(t_blk_edges) - 1} blocks x {len(f_grp_edges) - 1} groups)',
                               'likelihood grid / time mean per group']):
    ax.set_title(title, fontsize=10)
for ax in axes[1]:
    ax.set_xlabel('time [days]')
fig.tight_layout()
fig.savefig(OUT / 'wdm_timefreq.png', dpi=150)
print(f'pixels: {data.Nt} x {data.Nf} per channel (dT = {dT / 3600:.0f} h, dF = {dF * 1e6:.1f} uHz), '
      f'{len(band)} in [{cfg["f1"]}, {cfg["f2"]}] Hz; likelihood grid {Ybar.shape[1]} x {Ybar.shape[2]}, '
      f'pixels per group nu = {int(nu.min())}..{int(nu.max())}')

## 2. Sampling

eryn with reversible jump over the atoms of each channel (`mod_A`, `mod_E`) and a stretch move on the 8 spectral
parameters (`alpha, amp, fknee, fr1, fr2` of the galaxy, `A, P` of the noise, `level_E`). Priors: atom centre
$t_{\rm frac}\in[0,1]$, width 5–120 d (log-uniform), amplitude $\pm3$, $K\in[0,16]$ uniform, `level_E` $\in[-3,3]$, the
spectral bounds of [config/sources.yaml](../../30_days/config/sources.yaml). 32 walkers × 4 temperatures, 2000 burn-in + 6000 stored steps.

In [ ]:
res_path = OUT / 'chains.npz'
if RERUN or not res_path.exists():
    I.run('wavelet', 'run1', CFG)
r = dict(np.load(res_path, allow_pickle=True))
nsteps, nw = int(r['nsteps']), int(r['nwalkers'])
print(f"runtime {float(r['runtime_s']) / 60:.1f} min, {nsteps} steps x {nw} walkers (cold chain)")
print('acceptance (cold):', np.round(np.mean(r['acceptance_fraction']), 3),
      '| RJ acceptance (cold):', np.round(np.mean(r['rj_acceptance_fraction']), 3))

## 3. Convergence

Autocorrelation times (larger of a Sokal-windowed ACF and batch means), effective sample sizes and stuck walkers
([diagnostics.py](../../common/diagnostics.py)); then the number of atoms and the log-likelihood over the stored steps.

In [ ]:
D.report(r, post, np.array([60., 180., 300.]) * DAY)
print()
ll = r['log_like'].reshape(nsteps, nw)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for c, col in zip(CH, [C_POST, C_SECOND]):
    axes[0].plot(r[f'mod_{c}_nleaves'].reshape(nsteps, nw).mean(1), color=col, label=f'mean K, {c} (true {len(inj.atoms[c])})')
axes[0].set_xlabel('stored step (after burn-in)'); axes[0].set_ylabel('mean K over walkers'); axes[0].legend(fontsize=8)
axes[1].plot(np.median(ll, 1) - float(r['loglike_true']), color=C_POST, label='median over walkers')
axes[1].fill_between(np.arange(nsteps), *(np.percentile(ll, [5, 95], 1) - float(r['loglike_true'])), color=C_BAND, alpha=0.4, lw=0, label='5-95%')
axes[1].axhline(0, color=C_TRUTH, lw=1, label='at the injection')
axes[1].set_xlabel('stored step'); axes[1].set_ylabel(r'$\ln L - \ln L_{\rm inj}$'); axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'convergence.png', dpi=150)
d = ll.ravel() - float(r['loglike_true'])
n_free = 8 + 3 * sum(len(inj.atoms[c]) for c in CH)
print(f'ln L - ln L_inj: median {np.median(d):+.2f}, max {d.max():+.2f}  '
      f'(for ~{n_free} free parameters a typical posterior sample sits ~{n_free / 2:.0f} below the max)')

### Exploration of K, including the burn-in

eryn runs **32 walkers in each of 4 temperatures (128 in total)**; only the 32 walkers of the cold chain
($\beta = 1$) sample the posterior, and those are what every other plot uses. The hotter chains only help mixing
through swaps. `chains.npz` has no burn-in (eryn discards it), so the sampler was rerun with the same seeds and the
burn-in stored (`RUN_K_TRACE = True`, `outputs/run1/k_trace.npz`). The flag
`matches_chains` checks that the stored part of the rerun is the same chain as `chains.npz`.

Per step: the **band from the minimum to the maximum K over the 32 cold walkers**, the 25–75% range, and the mean.
All walkers start at K = 0. Right: the first 500 steps.

In [ ]:
kt_path = Path(I.k_trace_path('run1', CFG))
if RUN_K_TRACE and not kt_path.exists():
    I.run_k_trace('wavelet', 'run1', config=CFG)
if not kt_path.exists():
    print('no k_trace.npz yet: set RUN_K_TRACE = True to make it')
else:
    kt = dict(np.load(kt_path))
    burn, nw_t, ntemps = int(kt['burnin']), int(kt['nwalkers']), int(kt['ntemps'])
    print(f"{nw_t} walkers x {ntemps} temperatures = {nw_t * ntemps} walkers; posterior from the {nw_t} cold walkers")
    print(f"{burn} burn-in + {int(kt['nsteps'])} stored steps; rerun reproduces chains.npz: {bool(kt.get('matches_chains'))}")

    fig, axes = plt.subplots(2, 2, figsize=(11, 5.6), sharey='row', gridspec_kw={'width_ratios': [2.2, 1]})
    for row, (c, kt_true) in enumerate(zip(CH, kt['K_true'])):
        K = kt[f'K_{c}'][:, 0].astype(float)              # cold chain: (steps, walkers)
        step = np.arange(1, len(K) + 1)
        lo, hi = K.min(1), K.max(1)
        q25, q75 = np.percentile(K, [25, 75], axis=1)
        reach = np.argmax(hi >= kt_true) + 1                  # first step some walker has the true K
        left0 = np.argmax(lo >= 1) + 1                        # first step every walker has at least one atom
        for ax, sl, xmax in zip(axes[row], [slice(None), slice(0, 500)], [len(K), 500]):
            ax.fill_between(step[sl], lo[sl], hi[sl], step='mid', color=C_BAND, alpha=0.45, lw=0, label='min–max over cold walkers')
            ax.fill_between(step[sl], q25[sl], q75[sl], step='mid', color=C_POST, alpha=0.35, lw=0, label='25–75%')
            ax.plot(step[sl], K[sl].mean(1), color=C_POST, lw=1.2, label='mean')
            ax.axhline(kt_true, color=C_SECOND, ls='--', lw=1.2, label='true K')
            ax.axvspan(0, min(burn, xmax), color=C_MUTED, alpha=0.10, lw=0)
            ax.set_xlim(0, xmax)
        axes[row, 0].text(burn / 2, 0.95, 'burn-in', transform=axes[row, 0].get_xaxis_transform(), ha='center',
                          va='top', fontsize=8, color=C_MUTED)
        axes[row, 0].set_ylabel(f'K, channel {c}')
        print(f'{c}: K range over cold walkers, burn-in {int(lo[:burn].min())}-{int(hi[:burn].max())}, '
              f'stored {int(lo[burn:].min())}-{int(hi[burn:].max())}; true K first reached at step {reach}, '
              f'every walker off K = 0 from step {left0}; mean K: last 500 burn-in steps {K[burn - 500:burn].mean():.2f}, stored {K[burn:].mean():.2f}; '
              f'fraction of stored steps with the true K inside [min, max]: '
              f'{np.mean((lo[burn:] <= kt_true) & (kt_true <= hi[burn:])):.3f}')
    for ax in axes[1]:
        ax.set_xlabel('step (burn-in included)')
    fig.legend(*axes[0, 0].get_legend_handles_labels(), fontsize=8, ncol=4, loc='lower center', bbox_to_anchor=(0.5, -0.04))
    axes[0, 1].set_title('first 500 steps', fontsize=9)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(OUT / 'K_trace_burnin.png', dpi=150, bbox_inches='tight')

## 4. Posterior on the number of atoms

With a uniform prior on $K$, posterior odds are Bayes factors: $\ln\mathcal B(K, K') = \ln p(K|d) - \ln p(K'|d)$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
for ax, c in zip(axes, CH):
    K, p = r[f'mod_{c}_bayes_K'], r[f'mod_{c}_bayes_posterior']
    ax.bar(K, p, color=[C_SECOND if k == len(inj.atoms[c]) else C_POST for k in K])
    ax.set_xlabel('K'); ax.set_title(f'channel {c}: true K = {len(inj.atoms[c])} (orange)')
    ax.set_xlim(-0.6, post.max_wavelets + 0.6)
    print(c, ' '.join(f'p(K={k})={q:.3f}' for k, q in zip(K, p) if q > 1e-3),
          '| ln B(K_true vs best other) =', np.round(np.log(p[len(inj.atoms[c])]) - np.log(np.max(np.delete(p, len(inj.atoms[c])))), 2))
axes[0].set_ylabel('posterior probability')
fig.tight_layout()
fig.savefig(OUT / 'K_posterior.png', dpi=150)

## 5. Spectral parameters and `level_E`

In [ ]:
import corner
names = [str(n).split(':')[-1] for n in r['psd_names']]
fig = corner.corner(r['psd_chain'], labels=names, truths=r['psd_injected'], truth_color=C_SECOND, color=C_POST,
                    quantiles=[0.05, 0.5, 0.95], show_titles=True, title_fmt='.3f', bins=30, smooth=0.9)
fig.savefig(OUT / 'corner_psd.png', dpi=110)

print(f"{'param':8s} {'truth':>9s} {'median':>9s} {'5%':>9s} {'95%':>9s}  in 90%?  z")
for n, x, t in zip(names, r['psd_chain'].T, r['psd_injected']):
    q = np.percentile(x, [5, 50, 95])
    print(f'{n:8s} {t:9.3f} {q[1]:9.3f} {q[0]:9.3f} {q[2]:9.3f}  {str(q[0] <= t <= q[2]):5s}  {(q[1] - t) / np.std(x):+.2f}')

## 6. The atoms themselves

Atoms are exchangeable, so they are compared after sorting by centre, using only the samples with $K = K_{\rm true}$.
Atoms are not individually unique (an atom much wider than its neighbour trades off against it), so this plot is a
consistency check; §7 is the real test.

In [ ]:
C_ATOM = list(plt.cm.tab10.colors)
fig, axes = plt.subplots(2, 3, figsize=(12, 6.2))
labels = ['centre [d]', r'$\log_{10}$ width [d]', 'amplitude']
for row, c in enumerate(CH):
    kt = len(inj.atoms[c])
    sel = r[f'mod_{c}_nleaves'] == kt
    A = r[f'mod_{c}_atoms'][sel]
    M = r[f'mod_{c}_inds'][sel]
    A = np.stack([a[m] for a, m in zip(A, M)])                      # (n, kt, 3)
    A = np.take_along_axis(A, np.argsort(A[:, :, 0], 1)[:, :, None], 1)
    A[:, :, 0] *= inj.wm.T / DAY
    truth = inj.atoms[c][np.argsort(inj.atoms[c][:, 0])].copy()
    truth[:, 0] *= inj.wm.T / DAY
    print(f'channel {c}: {sel.sum()} samples with K = {kt}')
    for j in range(kt):
        for p in range(3):
            ax = axes[row, p]
            ax.hist(A[:, j, p], bins=50, histtype='step', color=C_ATOM[j % len(C_ATOM)], density=True, label=f'atom {j + 1}')
            ax.axvline(truth[j, p], color=C_ATOM[j % len(C_ATOM)], ls='--', lw=1.2)
            ax.set_xlabel(labels[p])
        q = np.percentile(A[:, j], [5, 50, 95], axis=0)
        print('   atom', j + 1, ' '.join(f'{l.split(" ")[0]}: {t:.2f} [{a:.2f}, {b:.2f}]' for l, t, a, b in zip(['centre', 'log10w', 'amp'], truth[j], q[0], q[2])))
    axes[row, 0].set_ylabel(f'channel {c}')
axes[0, 0].legend(fontsize=8)
fig.suptitle('atoms at K = K_true (dashed: injected)')
fig.tight_layout()
fig.savefig(OUT / 'atoms.png', dpi=150)

## 7. The recovered modulation

What is plotted is the **galactic power relative to the injection**,
$$10^{\,\Delta\log_{10}A}\,P_c(t), \qquad \Delta\log_{10}A = \texttt{amp} - \texttt{amp}_{\rm inj},$$
where `amp` $=\log_{10}A_{\rm gal}$ is the galactic amplitude of the posterior sample and `amp`$_{\rm inj}$ the injected
(effective) value (printed in section 1). Why not $P_c(t)$ alone: the galactic power in the likelihood is $A_{\rm gal}\,P_c(t)$, and
only that product is measured. A very wide atom is a constant in $\ln P_c$ over the window, so it can raise $P_c(t)$ while
`amp` drops by the same factor, leaving the data unchanged. $P_c(t)$ alone is therefore not identified; the product is.
Dividing by the injected amplitude puts it on the scale of the injected $P_c(t)$, so the truth is the dashed curve.
Bands are 50% and 90% over all K. Lower panels: fractional deviation from the injection.

In [ ]:
draws, t = r['modulation_draws'], r['modulation_times'] / DAY
q = np.percentile(draws, [5, 25, 50, 75, 95], axis=0)
P_true = r['P_true']
eb = cfg['wavelet']['edge_bins']
inside = slice(eb, len(t) - eb)

fig, axes = plt.subplots(2, 2, figsize=(11, 5.4), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
for i, c in enumerate(CH):
    ax, axr = axes[0, i], axes[1, i]
    ax.fill_between(t, q[0, i], q[4, i], color=C_BAND, alpha=0.45, lw=0, label='90%')
    ax.fill_between(t, q[1, i], q[3, i], color=C_POST, alpha=0.45, lw=0, label='50%')
    ax.plot(t, q[2, i], color=C_POST, label='median')
    ax.plot(t_fine / DAY, np.asarray(inj.P(t_fine, i)), color=C_TRUTH, ls='--', lw=1.2, label='injected')
    axr.fill_between(t, q[0, i] / P_true[i] - 1, q[4, i] / P_true[i] - 1, color=C_BAND, alpha=0.45, lw=0)
    axr.fill_between(t, q[1, i] / P_true[i] - 1, q[3, i] / P_true[i] - 1, color=C_POST, alpha=0.45, lw=0)
    axr.axhline(0, color=C_TRUTH, ls='--', lw=1)
    for a in (ax, axr):
        a.axvspan(t[0], t[eb], color=C_MUTED, alpha=0.12, lw=0)
        a.axvspan(t[-eb], t[-1], color=C_MUTED, alpha=0.12, lw=0)
    ax.set_title(f'channel {c}')
    axr.set_xlabel('time [days]')
    cover = np.mean((q[0, i, inside] <= P_true[i, inside]) & (P_true[i, inside] <= q[4, i, inside]))
    width = np.median((q[4, i, inside] - q[0, i, inside]) / P_true[i, inside])
    print(f'{c}: injection inside the 90% band at {100 * cover:.0f}% of analysed time bins; median 90% width = {100 * width:.1f}% of P')
axes[0, 0].set_ylabel(r'$10^{\,\mathrm{amp}-\mathrm{amp}_{\rm inj}}\,P_c(t)$')
axes[1, 0].set_ylabel(r'$P/P_{\rm inj} - 1$')
axes[0, 0].legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(OUT / 'modulation_recovery.png', dpi=150)

### 7b. The galactic power the data measure

The quantity above uses `amp` alone as the galactic normalisation. But the data measure the galactic **PSD** in the
band, $A_{\rm gal}\,S_{\rm shape}(f;\alpha, f_{\rm knee}, f_1, f_2)\,P_c(t)$, and `amp` trades off against the shape
parameters (a higher `amp` with a steeper or lower-cut spectrum gives the same power where the galaxy dominates). So
here each posterior sample's full galactic spectrum, relative to the injected one, multiplies $10^{\Delta\log_{10}A}P_c(t)$
in place of $10^{\Delta\log_{10}A}$, averaged over the frequency groups with the Fisher weight of a galactic-power scale,
$w_g \propto \nu_g\,[S_{\rm gal}/(S_{\rm gal}+S_n)]^2$ at the injection:

$$\frac{\sum_g w_g\,S_{\rm gal}(f_g;\theta)}{\sum_g w_g\,S_{\rm gal}(f_g;\theta_{\rm inj})}\cdot\frac{P_c(t;\theta)}{P_c(t;\theta_{\rm inj})}.$$

As a cross-check, the profile of the likelihood over one galactic-power scale per channel, with every other parameter
at the injection, is also printed.

In [ ]:
import jax
import jax.numpy as jnp
from bahamas.wavelet.sampler import _sampled
n_all = len(r['psd_chain'])
idx = np.random.default_rng(0).choice(n_all, size=min(1000, n_all), replace=False)   # the draws of modulation_draws
theta_inj = np.array([p.value for p in _sampled(post.psd_params)], float)
names_psd = [p.name for p in _sampled(post.psd_params)]
Sg0, Sn0, _ = post.spectra(jnp.asarray(theta_inj))
Sg0, Sn0 = np.asarray(Sg0), np.asarray(Sn0)
Sg = np.asarray(jax.vmap(lambda th: post.spectra(th)[0])(jnp.asarray(r['psd_chain'][idx])))
ok = Sg0 > 1e-3 * Sg0.max()
R = np.where(ok, Sg / np.where(ok, Sg0, 1.), 0.)
d_amp = 10 ** (r['psd_chain'][idx, names_psd.index('amp')] - theta_inj[names_psd.index('amp')])
nu_g = np.asarray(like.nu)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for i, c in enumerate(CH):
    w = np.where(ok, nu_g * (Sg0 / (Sg0 + Sn0[i])) ** 2, 0.)
    full = r['modulation_draws'][:, i, :] / P_true[i] / d_amp[:, None] * (R @ (w / w.sum()))[:, None]
    for lab, m, col in ((r'$10^{\Delta\log_{10}A}P_c$ (above)', r['modulation_draws'][:, i, :] / P_true[i], C_MUTED),
                        (r'$S_{\rm gal}(f)\,P_c$, Fisher-weighted', full, C_POST)):
        qq = np.percentile(m, [5, 50, 95], 0)
        axes[i].fill_between(t, qq[0] - 1, qq[2] - 1, color=col, alpha=0.3, lw=0)
        axes[i].plot(t, qq[1] - 1, color=col, label=lab)
        qs = qq[:, inside]
        print(f'{c}: {lab:42s} median offset {100 * (np.median(qs[1]) - 1):+.1f}%, injection inside the 90% band at '
              f'{100 * np.mean((qs[0] <= 1) & (1 <= qs[2])):.0f}% of analysed bins, median 90% width {100 * np.median(qs[2] - qs[0]):.1f}%')
    axes[i].axhline(0, color=C_TRUTH, ls='--', lw=1)
    axes[i].set_title(f'channel {c}'); axes[i].set_xlabel('time [days]')
axes[0].set_ylabel('relative to injection'); axes[0].legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(OUT / 'modulation_power.png', dpi=150); plt.show()

# profile likelihood over one galactic-power scale per channel, everything else at the injection
def ll_at(theta):
    atoms, masks = [], []
    for name in post.channel_names:
        a = np.zeros((1, post.max_wavelets, 3)); m = np.zeros((1, post.max_wavelets), bool)
        k = len(inj.atoms[name]); a[0, :k], m[0, :k] = inj.atoms[name], True
        atoms.append(a); masks.append(m)
    pad = post.batch_size or 1
    rep = lambda x: jnp.repeat(jnp.asarray(x), pad, axis=0)
    return float(post.batch(rep(jnp.asarray(theta)[None]), jnp.zeros((pad, 0)), tuple(map(rep, atoms)), tuple(map(rep, masks)))[0])
ia, ie = names_psd.index('amp'), names_psd.index('level_E')
grid = np.linspace(-0.08, 0.08, 33)
for c in CH:
    vals = []
    for d in grid:
        th = theta_inj.copy()
        if c == 'E':
            th[ie] += d
        else:
            th[ia] += d / np.log(10); th[ie] -= d
        vals.append(ll_at(th))
    a2, a1, _ = np.polyfit(grid, vals, 2)
    print(f'{c}: profile-likelihood galactic-power scale = {np.exp(-a1 / (2 * a2)):.3f} +- {1 / np.sqrt(-2 * a2):.3f}')

## 8. Summary

Compare with the 30-day run 1: [run1_inject_recover.ipynb](../../30_days/run1/run1_inject_recover.ipynb) and
[reports/WAVELET_INJECTION_RUN1.md](../../../reports/WAVELET_INJECTION_RUN1.md).